# Day 8 — Your First Machine Learning Model: Simple Linear Regression

**Duration:** ~1.5 to 2 hours
**Libraries:** NumPy · Pandas · Matplotlib · Seaborn · Scikit-learn

---

### What are we doing today?

You've explored data with EDA. Now we answer the next question:

> *"Can we use data to **predict** something we don't know yet?"*

Today you will build your **very first ML model** — a Simple Linear Regression —
that predicts a student's exam score based on how many hours they studied.

No scary math. No complex setup. Just clear steps, working code, and real intuition.

---

### The Big Picture — What is Machine Learning?

```
  TRADITIONAL PROGRAMMING          MACHINE LEARNING
  ───────────────────────          ────────────────
  Rules + Data → Answers           Data + Answers → Rules
  
  You write the logic.             The machine finds the pattern.
```

Today we'll let the machine find the pattern between **study hours** and **exam scores**.


## Step 1 — Import Libraries

These are the four tools we'll use today.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make plots look clean
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (8, 5)

print("✅ All libraries imported successfully!")

## Step 2 — Create the Dataset

We'll use a simple dataset: **study hours vs. exam scores** for 30 students.
No need to download anything — we create it right here.


In [ ]:
# Seed for reproducibility — same random numbers every time
np.random.seed(42)

# 30 students, studying between 1 and 10 hours
study_hours = np.round(np.random.uniform(1, 10, 30), 1)

# Exam score = roughly 5 * hours + some noise (real life isn't perfect!)
noise = np.random.normal(0, 5, 30)          # small random variation
exam_scores = np.round(5 * study_hours + 30 + noise, 1)
exam_scores = np.clip(exam_scores, 0, 100)  # keep scores between 0 and 100

# Put it in a DataFrame
df = pd.DataFrame({
    "study_hours": study_hours,
    "exam_score":  exam_scores
})

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
# Quick summary statistics
print("=== Dataset Summary ===")
print(df.describe().round(2))

## Step 3 — Explore the Data (EDA)

Before building any model, **always look at your data first**.
You already know how to do this from your EDA sessions!


In [ ]:
# Distribution of each column
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["study_hours"], bins=10, color="#7c6af7", edgecolor="white")
axes[0].set_title("Distribution of Study Hours", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Study Hours")
axes[0].set_ylabel("Number of Students")

axes[1].hist(df["exam_score"], bins=10, color="#3ecfb2", edgecolor="white")
axes[1].set_title("Distribution of Exam Scores", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Exam Score")
axes[1].set_ylabel("Number of Students")

plt.tight_layout()
plt.show()

In [ ]:
# The most important plot: does a relationship exist?
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="study_hours", y="exam_score",
                color="#7c6af7", s=80, edgecolor="white")
plt.title("Study Hours vs Exam Score", fontsize=14, fontweight='bold')
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")
plt.show()

# What do you notice? Does more study = higher score?

In [ ]:
# Confirm with correlation
corr = df["study_hours"].corr(df["exam_score"])
print(f"Correlation between study hours and exam score: {corr:.2f}")
print()
if corr > 0.7:
    print("✅ Strong positive correlation — a linear model should work well!")
elif corr > 0.4:
    print("⚠️  Moderate correlation — linear model may work, check the fit.")
else:
    print("❌ Weak correlation — linear regression may not be the right model.")

## Step 4 — What IS Linear Regression?

Before we code it, let's build the intuition.

### The core idea

Linear regression draws **the best possible straight line** through your data points.

```
  exam_score = m × study_hours + b

       m = slope     (how much score increases per extra hour of study)
       b = intercept (predicted score if study_hours = 0)
```

The model learns `m` and `b` from your training data.
Once it knows those two numbers, it can predict scores for **new students it has never seen**.

### What does "best line" mean?
The line that makes the **smallest total error** — meaning the vertical
distances between the line and the actual data points are minimised.
These distances are called **residuals**.

```
      actual score  ●
                     \  ← residual (error)
                      ✕  ← predicted score (on the line)
                      
  The model tries to make ALL residuals as small as possible.
```


## Step 5 — Prepare the Data

We need to split our data into:
- **Training set** — data the model *learns from* (80%)
- **Test set** — data the model has *never seen*, used to check how well it generalised (20%)

This is one of the most important habits in ML.
**Never test on data you trained on** — that's like giving students the exam paper beforehand!


In [ ]:
# X = input feature (what we know)
# y = target / label (what we want to predict)
X = df[["study_hours"]]   # double brackets → DataFrame (required by sklearn)
y = df["exam_score"]

print(f"X shape: {X.shape}  (feature matrix)")
print(f"y shape: {y.shape}  (target vector)")

In [ ]:
# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% goes to test
    random_state=42      # same split every time
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")
print()
print("Training data (first 5 rows):")
print(pd.concat([X_train, y_train], axis=1).head())

## Step 6 — Train the Model

This is the moment the machine **learns** from the data.
In scikit-learn, training is always just one line: `.fit()`


In [ ]:
# Create the model
model = LinearRegression()

# Train it — this is where learning happens!
model.fit(X_train, y_train)

print("✅ Model trained successfully!")
print()
print(f"  Slope (m)     : {model.coef_[0]:.2f}")
print(f"  Intercept (b) : {model.intercept_:.2f}")
print()
print(f"  → The model learned: exam_score = {model.coef_[0]:.2f} × study_hours + {model.intercept_:.2f}")
print()
print(f"  Interpretation: each extra hour of study adds ~{model.coef_[0]:.1f} points to the score.")

## Step 7 — Make Predictions

Now we use the trained model to predict scores for the **test set** students.
The model has never seen these students before!


In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

# Compare actual vs predicted
results = pd.DataFrame({
    "study_hours"     : X_test["study_hours"].values,
    "actual_score"    : y_test.values,
    "predicted_score" : np.round(y_pred, 1),
})
results["error"] = np.round(results["actual_score"] - results["predicted_score"], 1)
results = results.sort_values("study_hours").reset_index(drop=True)

print("Actual vs Predicted Scores:")
results

In [ ]:
# Try predicting for specific values
print("=== Custom Predictions ===")
test_hours = [[2], [5], [7], [9]]
for h in test_hours:
    pred = model.predict([h])[0]
    print(f"  Student studying {h[0]} hours → predicted score: {pred:.1f}")

## Step 8 — Visualise the Model

Let's draw the regression line on top of the data.
This is the most important plot in linear regression.


In [ ]:
plt.figure(figsize=(9, 5))

# Actual data points
sns.scatterplot(data=df, x="study_hours", y="exam_score",
                color="#7c6af7", s=80, edgecolor="white",
                label="Actual data", zorder=3)

# Regression line
x_line = np.linspace(df["study_hours"].min(), df["study_hours"].max(), 100).reshape(-1, 1)
y_line = model.predict(x_line)
plt.plot(x_line, y_line, color="#f0a040", linewidth=2.5, label="Regression line")

# Highlight test set predictions
plt.scatter(X_test["study_hours"], y_pred,
            color="#3ecfb2", s=100, marker="D",
            edgecolor="white", zorder=4, label="Test predictions")

plt.title("Linear Regression — Study Hours vs Exam Score",
          fontsize=14, fontweight='bold')
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot — how far off was each prediction?
plt.figure(figsize=(9, 4))
residuals = y_test.values - y_pred
plt.axhline(0, color="#f0a040", linewidth=1.5, linestyle="--", label="Zero error line")
plt.scatter(X_test["study_hours"], residuals,
            color="#7c6af7", s=80, edgecolor="white")
for i, (xv, rv) in enumerate(zip(X_test["study_hours"], residuals)):
    plt.vlines(xv, 0, rv, colors="#7c6af7", alpha=0.4, linewidth=1.5)
plt.title("Residuals (Actual − Predicted)", fontsize=13, fontweight='bold')
plt.xlabel("Study Hours")
plt.ylabel("Residual (Error)")
plt.legend()
plt.tight_layout()
plt.show()

print("Good model = residuals scattered randomly around 0 with no pattern.")

## Step 9 — Evaluate the Model

How do we know if our model is **good or bad**? We use metrics.

| Metric | Full Name | What it means | Ideal value |
|--------|-----------|---------------|-------------|
| MAE | Mean Absolute Error | Average error in score points | As low as possible |
| MSE | Mean Squared Error | Penalises big errors more | As low as possible |
| RMSE | Root Mean Squared Error | MAE but in original units | As low as possible |
| R² | R-squared | % of variation explained by model | Close to 1.0 |


In [ ]:
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("=== Model Evaluation ===")
print(f"  MAE  : {mae:.2f}  → predictions are off by ~{mae:.1f} points on average")
print(f"  MSE  : {mse:.2f}")
print(f"  RMSE : {rmse:.2f}  → same as MAE but penalises large errors more")
print(f"  R²   : {r2:.2f}   → the model explains {r2*100:.1f}% of score variation")
print()
if r2 >= 0.8:
    print("✅ Great fit! The model captures the trend well.")
elif r2 >= 0.5:
    print("⚠️  Decent fit — but there's room for improvement.")
else:
    print("❌ Poor fit — consider more features or a different model.")

## Step 10 — The Full ML Lifecycle (Summary)

You just completed your **first complete ML pipeline**. Here's what each step maps to:

```
┌─────────────────────────────────────────────────────────────────┐
│                    THE ML LIFECYCLE                             │
├────────────────────┬────────────────────────────────────────────┤
│  1. Define Problem │  "Can we predict exam scores?"             │
│  2. Collect Data   │  Created study_hours + exam_score dataset  │
│  3. Explore (EDA)  │  Histograms, scatter plot, correlation     │
│  4. Prepare Data   │  X/y split → train/test split             │
│  5. Train Model    │  model.fit(X_train, y_train)               │
│  6. Predict        │  model.predict(X_test)                     │
│  7. Evaluate       │  MAE, RMSE, R²                             │
│  8. Iterate        │  (Try new features, different models...)   │
└────────────────────┴────────────────────────────────────────────┘
```

Every ML project you ever work on — from fraud detection to LLMs —
follows these same steps. Today you did all of them.


## 🧪 Challenges — Go Further!

### Challenge 1 — Predict Your Own Score (Easy)
You study for **6.5 hours**. What does the model predict your score will be?


In [ ]:
# Challenge 1: Predict for 6.5 hours of study
# Your code here


### Challenge 2 — Add More Students (Easy)
Add 10 more rows to the dataset manually (make up realistic values).
Retrain the model and check if R² improves or gets worse.


In [ ]:
# Challenge 2: Add more data rows
extra_data = pd.DataFrame({
    "study_hours": [# your values here],
    "exam_score":  [# your values here]
})

# df_new = pd.concat([df, extra_data], ignore_index=True)
# Retrain and evaluate...


### Challenge 3 — Add a Second Feature (Medium)
Add a column `sleep_hours` (random between 4 and 9) to the dataset.
Train a **Multiple Linear Regression** model using both `study_hours`
and `sleep_hours` to predict `exam_score`.
Does the R² improve?

*Hint: Just add "sleep_hours" to X — sklearn handles multiple features automatically!*


In [ ]:
# Challenge 3: Multiple linear regression
np.random.seed(0)
df["sleep_hours"] = np.round(np.random.uniform(4, 9, len(df)), 1)

X_multi = df[["study_hours", "sleep_hours"]]
y_multi = df["exam_score"]

# Split, train, evaluate...
# Your code here


### Challenge 4 — Plot the Learning Curve (Hard)
Train the model on increasing amounts of training data (10%, 20%, ... 100%)
and plot how R² changes. This is called a **learning curve** and shows
whether your model needs more data.


In [ ]:
# Challenge 4: Learning curve
sizes = np.arange(0.1, 1.1, 0.1)
r2_scores = []

for size in sizes:
    # Hint: use train_test_split with test_size=(1-size) to get a subset
    # Train model, compute R² on fixed test set, append to r2_scores
    pass

# Plot r2_scores vs sizes
# Your code here


## Wrap-Up

Today you built your **first complete machine learning model**:

- ✅ Created and explored a dataset with EDA
- ✅ Understood what linear regression is doing (finding the best line)
- ✅ Split data into train and test sets — and understood *why*
- ✅ Trained a model with `model.fit()`
- ✅ Made predictions with `model.predict()`
- ✅ Evaluated using MAE, RMSE, and R²
- ✅ Visualised the regression line and residuals
- ✅ Mapped everything to the ML lifecycle

### Next session
We'll move toward **classification** — predicting categories (pass/fail,
spam/not spam) instead of continuous numbers — and introduce Decision Trees.
